In [1]:
from pathlib import Path
import pandas as pd
import sys
from catboost import CatBoostClassifier, Pool, EFstrType
from sklearn.metrics import roc_auc_score, log_loss, average_precision_score

# путь до корня проекта
PROJECT_ROOT = Path("../../").resolve()

# добавляем в PYTHONPATH
sys.path.append(str(PROJECT_ROOT))

from src.utils.preprocess import (
    preprocess_initial_dataset,
    split_dataset,
    remove_log_features,
    split_dataset_with_val,
    train_catboost_model,
    get_feature_importance_df
)


pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

DATA_PATH = Path("../../data/cleaned_dataset/final_dataset_from_notebooks.csv")
RAW_DATA_PATH = Path("../../data/raw/clean_dataset.csv")

In [2]:
df = pd.read_csv(DATA_PATH)
df_raw = pd.read_csv(RAW_DATA_PATH)
df = df.merge(
    df_raw[
        ["row_id", "lead_Стоимость доставки", "lead_Масса (гр)", "lead_Высота"]
    ],
    on="row_id",
    how="left",
)
df = remove_log_features(df)
del df_raw
# Приводим всё к нужному типу
df = preprocess_initial_dataset(df)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17966 entries, 0 to 18787
Columns: 110 entries, contact_pvz_code to contact_has_pvz_code
dtypes: datetime64[ns](2), float64(18), int64(64), object(26)
memory usage: 15.2+ MB


C:\Users\Михаил\AppData\Local\Temp\ipykernel_18728\3613829215.py:2: DtypeWarning: Columns (79,80,81) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv(RAW_DATA_PATH)


In [3]:
print(*df.columns.tolist(), sep="\n")

contact_pvz_code
lead_weight_gm
lead_responsible_user_id
lead_utm_group
lead_utm_referrer
lead_tags
lead_utm_source
lead_Квалификация лида
is_responsible_same
has_weight
contact_region_pvz
lead_utm_referrer_site
lead_has_roistat
lead_utm_id_1
lead_utm_id_2
lead_utm_id_3
lead_utm_device_type
lead_utm_site
lead_utm_position
lead_utm_reatrgeting_id
lead_utm_region_name
lead_is_utm_campaign_type_1
contact_Город
contact_region
lead_manager_category
lead_rate_is_warehouse_to_warehouse
lead_formname_has_value
lead_has_creation_date
lead_creation_date_week
lead_creation_date_month
lead_creation_date_quarter
lead_creation_date_dayofweek
lead_creation_date_sin
lead_creation_date_cos
sale_date_month
sale_date_quarter
sale_date_dayofweek
sale_date_week
sale_date_sin
sale_date_cos
lead_items_count
lead_total_quantity
lead_total_cost_from_composition
lead_has_health_supplement
lead_has_pillow
lead_has_mattress
lead_has_brace
lead_has_footwear
lead_has_accessory
lead_has_health_product
lead_categorie

In [4]:
def evaluate(model, X, y):
    y_refuse = (y == 0).astype(int)
    proba_refuse = model.predict_proba(X)[:, 0]
    return {
        "ROC_AUC": roc_auc_score(y_refuse, proba_refuse),
        "PR_AUC": average_precision_score(y_refuse, proba_refuse),
        "LogLoss": log_loss(y, model.predict_proba(X)),
    }

Обучаем датасет на всех признаках, чтобы посмотреть, какие признаки имеют наибольшее влияние на модель и есть ли среди них утечки

In [5]:
target = "buyout_flag"

X = df.drop(columns=[target, "sale_ts"])
y = df[target]

cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

train_pool = Pool(X, y, cat_features=cat_features)

model = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.1,
    verbose=50,
    random_seed=42,
)

model.fit(train_pool)

0:	learn: 0.6411631	total: 217ms	remaining: 1m 4s
50:	learn: 0.4057866	total: 3.92s	remaining: 19.1s
100:	learn: 0.3931512	total: 7.71s	remaining: 15.2s
150:	learn: 0.3814602	total: 11.6s	remaining: 11.5s
200:	learn: 0.3703261	total: 15.6s	remaining: 7.67s
250:	learn: 0.3586711	total: 19.6s	remaining: 3.82s
299:	learn: 0.3510226	total: 23.5s	remaining: 0us


In [6]:
fi = pd.DataFrame(
    {
        "feature": X.columns,
        "importance": model.get_feature_importance(train_pool),
    },
).sort_values(by="importance", ascending=False)

fi.head(20)

,feature,importance
72,lead_payment_type,5.439240
66,lead_has_length,4.756968
60,lead_created_ts,4.094471
59,timedelta_between_sale_and_creation,3.492951
42,lead_total_cost_from_composition,2.681770
68,lead_price,2.625012
5,lead_tags,2.494132
76,lead_mass_known,2.469882
16,lead_utm_device_type,2.424856
75,lead_group_quality,2.422282


In [7]:
df = df.sort_values("sale_ts")

X_train, y_train, X_test, y_test = split_dataset(df)

cat_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

model = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.1,
    verbose=0,
    cat_features=cat_features,
    random_state=42,
    )


model.fit(X_train, y_train)

metrics_full  = evaluate(model, X_test, y_test)
print("TIME SPLIT:", metrics_full)

TIME SPLIT: {'ROC_AUC': 0.6638482777589708, 'PR_AUC': 0.3887107605370326, 'LogLoss': 0.4305280496738976}


Удаляем из модели субъективную оценку менеджеров

In [8]:
df_reduced = df.drop(columns=["lead_Квалификация лида"])

X_train, y_train, X_test, y_test = split_dataset(df_reduced)

cat_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

model = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.1,
    verbose=0,
    cat_features=cat_features,
    random_state=42,
    )


model.fit(X_train, y_train)

metrics_reduced  = evaluate(model, X_test, y_test)
print("TIME SPLIT:", metrics_reduced)

TIME SPLIT: {'ROC_AUC': 0.6595262779282329, 'PR_AUC': 0.36881750358120025, 'LogLoss': 0.4403641513824959}


In [10]:
fi = get_feature_importance_df(model, X_train, y_train)

fi.head(20)

,feature,importance
67,lead_price,3.486347
71,lead_payment_type,3.128050
41,lead_total_cost_from_composition,3.083818
5,lead_tags,3.033921
58,timedelta_between_sale_and_creation,2.967541
59,lead_created_ts,2.864561
15,lead_utm_device_type,2.724714
74,lead_group_quality,2.588464
12,lead_utm_id_1,2.246633
105,lead_Высота,2.045763


Удаляем lead_group_quality так как потенциально содержит данные из будущего

In [11]:
df_reduced = df_reduced.drop(columns=["lead_group_quality"])

X_train, y_train, X_val, y_val, X_test, y_test = split_dataset_with_val(df_reduced)

model, metrics = train_catboost_model(
    X_train, y_train, X_val, y_val, X_test, y_test
)

print("Метрики после удаления lead_group_quality:", metrics)

0:	test: 0.4948507	best: 0.4948507 (0)	total: 60.9ms	remaining: 2m 1s
100:	test: 0.6347789	best: 0.6347789 (100)	total: 5.95s	remaining: 1m 51s
200:	test: 0.6358455	best: 0.6363337 (167)	total: 12.5s	remaining: 1m 51s
300:	test: 0.6377640	best: 0.6393592 (285)	total: 19s	remaining: 1m 47s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.639359198
bestIteration = 285

Shrink model to first 286 iterations.
Метрики после удаления lead_group_quality: {'ROC_AUC': 0.678800461238998, 'PR_AUC': 0.4062804622876623, 'LogLoss': 0.46344676536329676}


In [12]:
fi = get_feature_importance_df(model, X_train, y_train)

fi.head(20)

,feature,importance
71,lead_payment_type,11.889413
59,lead_created_ts,4.483890
104,lead_Высота,4.109042
15,lead_utm_device_type,3.778163
99,lead_height_known,3.052347
75,contact_to_lead_hours,2.867246
3,lead_utm_group,2.770163
84,lead_utm_campaign_missing,2.553224
67,lead_price,2.546965
6,lead_utm_source,2.138551


In [13]:
columns_to_drop = fi[fi["importance"] < 0.2]["feature"].to_list()
df_removed_unimportant = df_reduced.drop(columns=columns_to_drop)

In [14]:
X_train, y_train, X_val, y_val, X_test, y_test = split_dataset_with_val(
    df_removed_unimportant
)

model, metrics = train_catboost_model(
    X_train, y_train, X_val, y_val, X_test, y_test
)

print("Метрики после удаления хвоста fi:", metrics)

0:	test: 0.4919216	best: 0.4919216 (0)	total: 57.9ms	remaining: 1m 55s
100:	test: 0.6236221	best: 0.6267876 (84)	total: 5.65s	remaining: 1m 46s
200:	test: 0.6259418	best: 0.6275276 (152)	total: 11.7s	remaining: 1m 44s
300:	test: 0.6282397	best: 0.6307646 (282)	total: 18.1s	remaining: 1m 42s
400:	test: 0.6308983	best: 0.6312061 (372)	total: 24.8s	remaining: 1m 38s
500:	test: 0.6302982	best: 0.6316042 (431)	total: 31.3s	remaining: 1m 33s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.6316041505
bestIteration = 431

Shrink model to first 432 iterations.
Метрики после удаления хвоста fi: {'ROC_AUC': 0.6475509901828029, 'PR_AUC': 0.3375712025538554, 'LogLoss': 0.4891234562649383}


In [15]:
# baseline_pr_auc = 0.4062804622876623

# results = []

# for feature in fi["feature"]:
#     print(f"\nУдаляем признак: {feature}")

#     df_removed = df_reduced.drop(columns=[feature])

#     X_train, y_train, X_val, y_val, X_test, y_test = split_dataset_with_val(
#         df_removed
#     )

#     model, metrics = train_catboost_model(
#         X_train, y_train, X_val, y_val, X_test, y_test
#     )

#     pr_auc = metrics["PR_AUC"]

#     print(f"PR_AUC: {pr_auc:.6f}")

#     results.append((feature, pr_auc))

#     # проверка улучшения
#     if pr_auc > baseline_pr_auc:
#         print(f"✅ Улучшение! Удаление признака {feature} дало PR_AUC = {pr_auc:.6f}")

In [16]:
results_df = pd.DataFrame(results, columns=["feature", "pr_auc"])
results_df = results_df.sort_values(by="pr_auc", ascending=False)

print(results_df[results_df["pr_auc"] > baseline_pr_auc])

NameError: name 'results' is not defined

Удаляем `sale_hour`

In [ ]:
df_reduced = df_reduced.drop(columns=["sale_hour"])
X_train, y_train, X_val, y_val, X_test, y_test = split_dataset_with_val(
    df_reduced
)

model, metrics = train_catboost_model(
    X_train, y_train, X_val, y_val, X_test, y_test
)

print("Метрики после удаления признаков от sale_ts:", metrics)

0:	test: 0.5932611	best: 0.5932611 (0)	total: 70.9ms	remaining: 2m 21s
100:	test: 0.6309356	best: 0.6381838 (91)	total: 5.95s	remaining: 1m 51s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.6381838139
bestIteration = 91

Shrink model to first 92 iterations.
Метрики после удаления признаков от sale_ts: {'ROC_AUC': 0.6887393153351389, 'PR_AUC': 0.41769757566691534, 'LogLoss': 0.49658491346134326}


In [ ]:
fi = get_feature_importance_df(model, X_train, y_train)

fi.tail(50)

,feature,importance
91,utm_sky_brand,0.510259
84,lead_utm_campaign_grouped,0.499091
50,lead_has_linear_width,0.480939
23,lead_manager_category,0.417182
47,lead_has_accessory,0.401992
32,lead_creation_date_cos,0.350947
60,lead_created_hour,0.325130
29,lead_creation_date_quarter,0.293722
18,lead_utm_reatrgeting_id,0.286869
49,lead_categories_count,0.246833


In [77]:
baseline_pr_auc = 0.41769757566691534

for n in range(90, 29, -1):
    col_to_drop = fi["feature"].tail(n).to_list()

    df_test = df_reduced.drop(columns=col_to_drop)

    X_train, y_train, X_val, y_val, X_test, y_test = split_dataset_with_val(df_test)

    model, metrics = train_catboost_model(
        X_train, y_train, X_val, y_val, X_test, y_test
    )

    pr_auc = metrics["PR_AUC"]

    print(f"tail({n}) -> PR_AUC: {pr_auc:.6f}")

    if pr_auc > baseline_pr_auc:
        print("\nIMPROVEMENT FOUND")
        print(f"tail({n}) gives PR_AUC = {pr_auc:.6f} (baseline = {baseline_pr_auc})")
        break

0:	test: 0.5782315	best: 0.5782315 (0)	total: 43.2ms	remaining: 1m 26s
100:	test: 0.6193590	best: 0.6205655 (94)	total: 3.89s	remaining: 1m 13s
200:	test: 0.6247509	best: 0.6247540 (198)	total: 7.71s	remaining: 1m 9s
300:	test: 0.6303013	best: 0.6305127 (274)	total: 11.8s	remaining: 1m 6s
400:	test: 0.6292860	best: 0.6333361 (332)	total: 15.8s	remaining: 1m 3s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.6333361319
bestIteration = 332

Shrink model to first 333 iterations.
tail(90) -> PR_AUC: 0.366716
0:	test: 0.5737725	best: 0.5737725 (0)	total: 40.5ms	remaining: 1m 20s
100:	test: 0.6287870	best: 0.6325354 (65)	total: 3.82s	remaining: 1m 11s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.6325354403
bestIteration = 65

Shrink model to first 66 iterations.
tail(89) -> PR_AUC: 0.386414
0:	test: 0.5949185	best: 0.5949185 (0)	total: 39.1ms	remaining: 1m 18s
100:	test: 0.6437793	best: 0.6441758 (88)	total: 3.96s	remaining: 1m 14s
200:	test: 0.64

In [78]:
col_to_drop = fi["feature"].tail(61).to_list()

df_reduced = df_reduced.drop(columns=col_to_drop)
X_train, y_train, X_val, y_val, X_test, y_test = split_dataset_with_val(
    df_reduced
)

model, metrics = train_catboost_model(
    X_train, y_train, X_val, y_val, X_test, y_test
)

print("Метрики после удаления признаков от sale_ts:", metrics)

0:	test: 0.5457140	best: 0.5457140 (0)	total: 55.1ms	remaining: 1m 50s
100:	test: 0.6443406	best: 0.6453729 (84)	total: 5.31s	remaining: 1m 39s
200:	test: 0.6445645	best: 0.6482803 (130)	total: 10.3s	remaining: 1m 32s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.6482803011
bestIteration = 130

Shrink model to first 131 iterations.
Метрики после удаления признаков от sale_ts: {'ROC_AUC': 0.6851922181787407, 'PR_AUC': 0.41628202143501164, 'LogLoss': 0.46627536716179624}


In [79]:
X_train.columns.to_list()

['lead_weight_gm',
 'lead_utm_group',
 'lead_tags',
 'lead_utm_source',
 'lead_utm_referrer_site',
 'lead_utm_id_2',
 'lead_utm_id_3',
 'lead_utm_device_type',
 'lead_utm_position',
 'lead_formname_has_value',
 'lead_has_creation_date',
 'sale_date_week',
 'sale_date_sin',
 'sale_date_cos',
 'lead_items_count',
 'lead_total_cost_from_composition',
 'lead_discount_category',
 'lead_discount',
 'sale_dayofweek',
 'sale_month',
 'lead_created_ts',
 'lead_created_month',
 'lead_has_shipping_cost',
 'lead_shipping_cost',
 'lead_has_length',
 'lead_length',
 'lead_price',
 'lead_group_id',
 'width_cat',
 'width_is_missing',
 'lead_payment_type',
 'lead_delivery_type',
 'lead_mass_known',
 'contact_to_lead_hours',
 'contact_hour',
 'lead_category_freq',
 'lead_utm_campaign_missing',
 'utm_term_missing',
 'utm_sky_autotarget',
 'lead_group_missing',
 'problem_grouped',
 'lead_height_bin',
 'lead_Масса (гр)',
 'lead_Высота']